# 日期、时间与国际化

学习目标：区分时间点、日历日期和本地显示，选择国际化 API，并识别 Temporal 的支持条件。

前置知识：Number 与字符串、对象方法、数组回调、Unicode 码点与错误处理。

适用版本：Date 依据 ECMAScript 2025，Intl 依据 ECMA-402 第 12 版；示例使用 Node.js 24.11.0。Temporal 为超出本课程基线的补充。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。以下命令均从此目录运行；每个入口使用独立 Node.js 进程。

配套脚本：位于 scripts/18-dates-and-intl/。

1. [date-values.mjs](scripts/18-dates-and-intl/date-values.mjs)：时间戳、UTC 构造与无效日期。
2. [invalid-date-error.mjs](scripts/18-dates-and-intl/invalid-date-error.mjs)：无效日期不能转换成 ISO 文本。
3. [parse-and-zones.mjs](scripts/18-dates-and-intl/parse-and-zones.mjs)：标准解析和本地时间关系。
4. [date-arithmetic.mjs](scripts/18-dates-and-intl/date-arithmetic.mjs)：经过时长、溢出与月底截断。
5. [locale.mjs](scripts/18-dates-and-intl/locale.mjs)：区域标签与最终格式设置。
6. [date-format.mjs](scripts/18-dates-and-intl/date-format.mjs)：日期部件与固定时区。
7. [daylight-saving.mjs](scripts/18-dates-and-intl/daylight-saving.mjs)：跨夏令时的显示与经过时间。
8. [number-format.mjs](scripts/18-dates-and-intl/number-format.mjs)：数值的区域化显示。
9. [collation-and-plurals.mjs](scripts/18-dates-and-intl/collation-and-plurals.mjs)：区域排序与复数消息选择。
10. [segmentation.mjs](scripts/18-dates-and-intl/segmentation.mjs)：字素簇和词分段。
11. [other-formatters.mjs](scripts/18-dates-and-intl/other-formatters.mjs)：四种显示任务。
12. [duration-sign-error.mjs](scripts/18-dates-and-intl/duration-sign-error.mjs)：时长字段不能混合正负。
13. [temporal-support.mjs](scripts/18-dates-and-intl/temporal-support.mjs)：建立 Temporal 的能力边界。

## 1 Date 保存的是时间点

Date 内部保存从 1970-01-01T00:00:00Z 起的毫秒数，早于起点可为负数；有效范围为正负 8.64 × 10¹⁵ 毫秒，超出范围成为无效日期。它不保存创建时的时区标签。getTime 取毫秒时间戳，toISOString 输出 UTC 文本；时间戳单位不要与接口中常见的秒混用。

无参数 new Date() 和 Date.now() 读取当前时间；本章使用固定输入以便比较。多数字参数的 Date 构造器按本地时间解释，Date.UTC 则返回 UTC 毫秒数。两者月份索引都是 0–11，日从 1 起，年份 0–99 会映射到 1900–1999，不能把两位年份当作一般年份输入。

无效 Date 仍是对象；应以 Number.isNaN(date.getTime()) 判断。其 toISOString 会抛 RangeError，不能仅凭“创建时没报错”判断日期有效。

[date-values.mjs](scripts/18-dates-and-intl/date-values.mjs)：

```javascript
const epoch = new Date(0);
console.log(epoch.toISOString(), epoch.getTime());
const start = new Date(Date.UTC(2025, 0, 2, 3, 4, 5));
console.log(start.toISOString());
console.log(start.getUTCFullYear(), start.getUTCMonth(), start.getUTCDate());
console.log(new Date(Date.UTC(25, 0, 1)).getUTCFullYear());
console.log(Number.isNaN(new Date(NaN).getTime()));
console.log(Number.isNaN(new Date(8640000000000001).getTime()));

// 按本例输入运行，输出依次为：
// 1970-01-01T00:00:00.000Z 0
// 2025-01-02T03:04:05.000Z
// 2025 0 2
// 1925
// true
// true
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/date-values.mjs
```

[invalid-date-error.mjs](scripts/18-dates-and-intl/invalid-date-error.mjs)：

```javascript
new Date(NaN).toISOString();

// 独立运行：退出状态为 1；诊断包含 RangeError；Invalid time value。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/18-dates-and-intl/invalid-date-error.mjs
```

## 2 解析时明确偏移，读取时明确视角

标准日期时间字符串中，Z 表示 UTC，+08:00 等后缀表示相对 UTC 的偏移。无偏移的纯日期形式按 UTC 解释，而无偏移的日期加时间形式按宿主本地时间解释。不要把二者混用；非标准文本可进入实现自定的解析规则，业务输入应约定完整格式。

getHours 等本地方法与 getUTCHours 等 UTC 方法读取的是同一个时间点的不同表示。getTimezoneOffset 返回“UTC 减本地”的分钟偏移，符号与常见 +08:00 文本方向相反；它还可能随日期变化。下例以关系式检查本地解析，不硬编码本机所在时区。Date 的本地方法使用宿主默认时区，指定另一地区的显示要用 Intl.DateTimeFormat。

[parse-and-zones.mjs](scripts/18-dates-and-intl/parse-and-zones.mjs)：

```javascript
const withOffset = new Date("2025-01-02T08:00:00+08:00");
const utc = new Date("2025-01-02T00:00:00Z");
console.log(withOffset.getTime() === utc.getTime());
console.log(new Date("2025-01-02").toISOString());
const local = new Date("2025-01-02T08:00:00");
console.log(local.getFullYear(), local.getMonth(), local.getDate(), local.getHours());
const asUtc = Date.UTC(2025, 0, 2, 8);
console.log(local.getTime() === asUtc + local.getTimezoneOffset() * 60_000);
console.log(withOffset.getUTCHours());

// 按本例输入运行，输出依次为：
// true
// 2025-01-02T00:00:00.000Z
// 2025 0 2 8
// true
// 0
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/parse-and-zones.mjs
```

## 3 经过时长与日历计算

两个有效时间戳相减得到经过毫秒数。加 86,400,000 毫秒表示经过 24 小时；当地日历的“下一天同一钟点”在夏令时转换附近可能相隔 23 或 25 小时，业务应先决定所需含义。

Date 的 set 方法原地修改对象，越界的日、月会规范化。例如 1 月 31 日直接 setUTCMonth(1) 会把不存在的 2 月 31 日推进到 3 月，不能当作自动“月底对齐”。本例先设为目标月 1 日，再取目标月末与原日期的较小值，实现明确的月底截断规则。UTC 计算避开本地夏令时，但仍须自行定义月底业务规则。

[date-arithmetic.mjs](scripts/18-dates-and-intl/date-arithmetic.mjs)：

```javascript
const start = new Date("2025-01-31T00:00:00Z");
const oneDayLater = new Date(start.getTime() + 86_400_000);
console.log((oneDayLater - start) / 3_600_000);
const overflow = new Date(start);
overflow.setUTCMonth(1);
console.log(overflow.toISOString());
const clamped = new Date(start);
const originalDay = clamped.getUTCDate();
clamped.setUTCDate(1);
clamped.setUTCMonth(clamped.getUTCMonth() + 1);
const lastDay = new Date(Date.UTC(clamped.getUTCFullYear(),
  clamped.getUTCMonth() + 1, 0)).getUTCDate();
clamped.setUTCDate(Math.min(originalDay, lastDay));
console.log(clamped.toISOString(), start.toISOString());

// 按本例输入运行，输出依次为：
// 24
// 2025-03-03T00:00:00.000Z
// 2025-02-28T00:00:00.000Z 2025-01-31T00:00:00.000Z
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/date-arithmetic.mjs
```

## 4 区域设置与 Intl.Locale

Intl 属于 ECMA-402 标准接口。区域标签由语言、可选文字系统、地区和扩展组成；zh-Hans-CN-u-nu-latn 表示中文、简体、中国地区及拉丁数字系统。Intl.Locale 解析、组织标签，既不翻译内容，也不设置进程时区。

格式器通过 locales 和 options 选择行为；supportedLocalesOf 判断请求标签是否可支持而无需默认区域回退，resolvedOptions 查看最终采用的设置。省略设置时会用宿主默认值。Node.js 的 Intl 依赖 ICU 及其数据，定制构建的数据可能不完整；浏览器、ICU 和时区数据库版本不同也可能改变名称、空格或规则。展示文本不要当作跨环境的存储协议。

[locale.mjs](scripts/18-dates-and-intl/locale.mjs)：

```javascript
const locale = new Intl.Locale("zh-Hans-CN-u-nu-latn");
console.log(locale.language, locale.script, locale.region, locale.numberingSystem);
console.log(Intl.NumberFormat.supportedLocalesOf(["zh-CN", "en-US"]).join(","));
const format = new Intl.NumberFormat("en-US", { numberingSystem: "latn" });
const options = format.resolvedOptions();
console.log(options.locale, options.numberingSystem);

// 按本例输入运行，输出依次为：
// zh Hans CN latn
// zh-CN,en-US
// en-US latn
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/locale.mjs
```

## 5 日期显示、时区和夏令时

Intl.DateTimeFormat 把时间点按指定时区、日历和区域规则显示，不改变 Date。format 返回显示文本，formatToParts 返回带 type 与 value 的部件，适合按语义抽取日期字段，避免猜测分隔符。

America/New_York 是带历史规则的命名时区，并非固定 UTC−5。下面两个 UTC 时间点只差一小时，纽约墙上时钟却从 01:30 跳到 03:30。偏移和夏令时规则来自宿主时区数据，不是 Date 自行计算的地区政策。省略 timeZone 可能让同一程序在不同机器上显示不同日期。

[date-format.mjs](scripts/18-dates-and-intl/date-format.mjs)：

```javascript
const format = new Intl.DateTimeFormat("en-GB", {
  timeZone: "UTC", calendar: "gregory", numberingSystem: "latn",
  year: "numeric", month: "2-digit", day: "2-digit"
});
const date = new Date("2025-01-02T03:04:05Z");
console.log(format.format(date));
const parts = format.formatToParts(date);
console.log(["year", "month", "day"].map(type =>
  parts.find(part => part.type === type).value).join("-"));

// 按本例输入运行，输出依次为：
// 02/01/2025
// 2025-01-02
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/date-format.mjs
```

[daylight-saving.mjs](scripts/18-dates-and-intl/daylight-saving.mjs)：

```javascript
const before = new Date("2025-03-09T06:30:00Z");
const after = new Date("2025-03-09T07:30:00Z");
const clock = new Intl.DateTimeFormat("en-GB", {
  timeZone: "America/New_York", hourCycle: "h23",
  hour: "2-digit", minute: "2-digit"
});
console.log(clock.format(before), clock.format(after));
console.log((after - before) / 60_000);

// 按本例输入运行，输出依次为：
// 01:30 03:30
// 60
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/daylight-saving.mjs
```

## 6 数值格式、排序与复数类别

Intl.NumberFormat 支持十进制、百分比、货币和计量单位等显示。百分比会把 0.125 显示为 12.5%，不会改变原数值；格式化的舍入也不修复浮点运算误差。currency 是货币代码，unit 是支持的单位标识，需要与相应 style 配合。

Intl.Collator.compare 按区域比较字符串，只保证结果为负、零或正，不保证恰好为 −1 或 1；numeric: true 按数字片段比较“第2章”和“第10章”。Intl.PluralRules.select 返回复数类别，应用仍须提供该类别的实际句子；它不是翻译器。cardinal 表基数，ordinal 表序数，不同语言的类别和分界可能不同。

[number-format.mjs](scripts/18-dates-and-intl/number-format.mjs)：

```javascript
const percent = new Intl.NumberFormat("en-US", {
  style: "percent", maximumFractionDigits: 1
});
console.log(percent.format(0.125));
console.log(new Intl.NumberFormat("en-US", {
  style: "currency", currency: "USD"
}).format(1234.5));
console.log(new Intl.NumberFormat("en-US", {
  style: "unit", unit: "kilometer", unitDisplay: "long"
}).format(2));

// 按本例输入运行，输出依次为：
// 12.5%
// $1,234.50
// 2 kilometers
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/number-format.mjs
```

[collation-and-plurals.mjs](scripts/18-dates-and-intl/collation-and-plurals.mjs)：

```javascript
const compare = new Intl.Collator("en", { numeric: true }).compare;
console.log(["lesson10", "lesson2", "lesson1"].sort(compare).join(","));
console.log(Math.sign(compare("lesson2", "lesson10")));
const plural = new Intl.PluralRules("en", { type: "cardinal" });
const nouns = { one: "lesson", other: "lessons" };
for (const count of [0, 1, 2]) {
  console.log(count, plural.select(count), nouns[plural.select(count)]);
}
console.log(new Intl.PluralRules("en", { type: "ordinal" }).select(2));

// 按本例输入运行，输出依次为：
// lesson1,lesson2,lesson10
// -1
// 0 other lessons
// 1 one lesson
// 2 other lessons
// two
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/collation-and-plurals.mjs
```

## 7 按文字边界分段

Intl.Segmenter 按 grapheme、word 或 sentence 粒度分段，分别对应字素簇、词和句子。字素簇更接近用户感知的一个字符，可能由多个码点组成；它与字符串 length 的 UTF-16 码元数、for...of 的码点数不同。

segment 返回可迭代分段集合，各段的 index 是原字符串中的 UTF-16 索引。word 粒度还提供 isWordLike，可过滤空白和标点。分词依赖区域规则和实现数据，不能把它当作所有领域通用的自然语言理解。

[segmentation.mjs](scripts/18-dates-and-intl/segmentation.mjs)：

```javascript
const text = "e\u0301😀";
const graphemes = [...new Intl.Segmenter("en", {
  granularity: "grapheme"
}).segment(text)];
console.log(text.length, [...text].length, graphemes.length);
console.log(graphemes.map(part => part.index).join(","));
const words = new Intl.Segmenter("en", { granularity: "word" });
console.log([...words.segment("Read two chapters!")].filter(part =>
  part.isWordLike).map(part => part.segment).join("|"));

// 按本例输入运行，输出依次为：
// 4 3 2
// 0,2
// Read|two|chapters
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/segmentation.mjs
```

## 8 列表、相对时间、名称与时长

这些 Intl 格式器面向不同输入，不能互相替代。

| 原文名称 | 中文名称／含义 | 输入和用途 |
| --- | --- | --- |
| Intl.ListFormat | 列表格式器 | 将字符串列表按合取、析取或单位列表规则连接 |
| Intl.RelativeTimeFormat | 相对时间格式器 | 将带正负号的数量和单位写成“昨天”“三天后”等 |
| Intl.DisplayNames | 显示名称格式器 | 将语言、地区、货币等代码变成区域化名称 |
| Intl.DurationFormat | 时长格式器 | 将 hours、minutes 等字段组成的时长记录显示为文本 |

ListFormat 的列表成员须为字符串；RelativeTimeFormat 的负数表示过去，正数表示未来，numeric: auto 允许“昨天”等词，它不负责计算两个日期之差。DisplayNames 必须指定 type，输入代码应与类型匹配。

DurationFormat 已在 ECMA-402 第 12 版中，Node.js 24.11.0 支持。它格式化时长记录，不负责日历相加或时区转换；字段必须为整数，同一记录的非零字段不能正负混杂。digital 风格适合小时、分钟、秒显示，但不是任意毫秒时间戳的解释器。

[other-formatters.mjs](scripts/18-dates-and-intl/other-formatters.mjs)：

```javascript
console.log(new Intl.ListFormat("en", {
  style: "long", type: "conjunction"
}).format(["Map", "Set", "Date"]));
const relative = new Intl.RelativeTimeFormat("en", { numeric: "auto" });
console.log(relative.format(-1, "day"), relative.format(3, "day"));
console.log(new Intl.DisplayNames("en", { type: "region" }).of("CN"));
console.log(new Intl.DurationFormat("en", {
  style: "digital"
}).format({ hours: 1, minutes: 2, seconds: 3 }));

// 按本例输入运行，输出依次为：
// Map, Set, and Date
// yesterday in 3 days
// China
// 1:02:03
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/other-formatters.mjs
```

[duration-sign-error.mjs](scripts/18-dates-and-intl/duration-sign-error.mjs)：

```javascript
new Intl.DurationFormat("en").format({ hours: 1, minutes: -2 });

// 独立运行：退出状态为 1；诊断包含 RangeError；Invalid object。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/18-dates-and-intl/duration-sign-error.mjs
```

## 9 补充：Temporal 的职责和支持条件

Temporal 把不同时间概念分成不可变类型，减少 Date 把时间点和本地日历混在同一对象中的歧义。下面是用途定位，不是本机已提供这些类型的承诺。

| 原文名称 | 中文名称／含义 | 适用数据 |
| --- | --- | --- |
| Temporal.Instant | 绝对时间点 | 与特定时区无关的时间线位置 |
| Temporal.PlainDate | 无时区日期 | 生日、仅按日历记录的日期 |
| Temporal.PlainTime | 无时区钟点 | 每日营业时间等 |
| Temporal.PlainDateTime | 无时区日期时间 | 尚未绑定具体时区的日历和钟点 |
| Temporal.ZonedDateTime | 带时区日期时间 | 同时关联时间点、时区与日历 |
| Temporal.Duration | 时长 | 年、月、日、时等单位组成的量 |

标准状态与宿主支持必须分开：截至 2026-09-12 核查，TC39 已完成提案表将 Temporal 列为 Stage 4、预计出版年份 2027；它不属于本课程 ECMA-262 第 16 版基线。固定的 Node.js 24.11.0 默认进程中 globalThis.Temporal 为 undefined，因此本节用能力检测建立调用边界，不直接调用不存在的构造器，也不借安装或实验参数改变运行基线。采用支持 Temporal 的宿主或另行选择 polyfill 时，还须核查相应实现和数据条件。

[temporal-support.mjs](scripts/18-dates-and-intl/temporal-support.mjs)：

```javascript
const temporalAvailable = typeof globalThis.Temporal !== "undefined";
console.log(temporalAvailable);
console.log(temporalAvailable ? "可进一步核查所需 Temporal 类型" : "当前使用 Date 与 Intl");

// 按本例输入运行，输出依次为：
// false
// 当前使用 Date 与 Intl
```

Step 1：运行本节示例。

```bash
node scripts/18-dates-and-intl/temporal-support.mjs
```

## 本章小结

- Date 保存毫秒时间点；解析、显示与日历计算必须明确偏移、时区和月底规则。
- Intl 按任务选择格式器，区域化文本与时区规则依赖环境数据。
- Temporal 的标准进展不等于固定宿主已提供实现，调用前先建立能力边界。

## 练习

1. 把 2025-01-02T08:00:00+08:00 与 UTC 文本比较。可核对标准：毫秒时间戳相等，转 UTC 后为 2025-01-02T00:00:00.000Z。
2. 按本章的月底截断规则把 2024-01-31 增加一个 UTC 月。可核对标准：结果为 2024-02-29，原 Date 保持不变。
3. 将同一时间点分别按 UTC 与 America/New_York 显示，并使用 word 粒度分割一个英文短句。可核对标准：显式填写 timeZone；解释时间戳不变；分段结果排除标点和空白。
4. 为“距开始还有 3 天”和“课程耗时 1 小时 2 分钟”选择 API。可核对标准：分别使用 RelativeTimeFormat 与 DurationFormat，并指出它们都不自动计算日期差。

## 参考与引用来源

- TC39 官方文档：[ECMAScript 2025 §21.4 Date 时间值、构造、解析及原型方法](https://tc39.es/ecma262/2025/multipage/numbers-and-dates.html#sec-date-objects)；[Temporal 类型、不可变值及日期时间职责](https://tc39.es/proposal-temporal/docs/)。
- Ecma International：[ECMA-402 第 12 版，§8 Intl 及 §9 区域协商](https://402.ecma-international.org/12.0/#intl-object)；[§10 Collator](https://402.ecma-international.org/12.0/#collator-objects)；[§11 DateTimeFormat 与时区](https://402.ecma-international.org/12.0/#datetimeformat-objects)；[§12 DisplayNames](https://402.ecma-international.org/12.0/#intl-displaynames-objects)；[§13 DurationFormat](https://402.ecma-international.org/12.0/#durationformat-objects)；[§14 ListFormat](https://402.ecma-international.org/12.0/#listformat-objects)；[§15 Locale](https://402.ecma-international.org/12.0/#locale-objects)；[§16 NumberFormat](https://402.ecma-international.org/12.0/#numberformat-objects)；[§17 PluralRules](https://402.ecma-international.org/12.0/#pluralrules-objects)；[§18 RelativeTimeFormat](https://402.ecma-international.org/12.0/#relativetimeformat-objects)；[§19 Segmenter](https://402.ecma-international.org/12.0/#segmenter-objects)。
- Node.js：[24.11.0：ICU 构建与数据支持](https://nodejs.org/download/release/v24.11.0/docs/api/intl.html)。
- GitHub 上的 TC39 提案清单：[Temporal 的 Stage 4 与预计出版年份；核查日期 2026-09-12](https://github.com/tc39/proposals/blob/main/finished-proposals.md)。